Install required libraries

In [12]:
!pip install -q transformers accelerate sentencepiece scikit-learn

Import libraries and load tokenizer

In [13]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

indicbert_name = "ai4bharat/IndicBERTv2-MLM-only"

indicbert_tokenizer = AutoTokenizer.from_pretrained(indicbert_name)

MAX_LENGTH = 128

print("IndicBERT v2 tokenizer loaded successfully!")
print("Vocabulary size:", indicbert_tokenizer.vocab_size)
print("GPU available:", torch.cuda.is_available())

IndicBERT v2 tokenizer loaded successfully!
Vocabulary size: 250000
GPU available: True


Load the dataset

In [14]:
multilingual_df = pd.read_csv(
    "/content/GWSD_Multilingual.tsv",
    sep="\t"
)

print("Dataset shape:", multilingual_df.shape)
print("\nColumns:")
print(multilingual_df.columns.tolist())

multilingual_df.head()

Dataset shape: (2300, 20)

Columns:
['Unnamed: 0', 'round', 'batch', 'sent_id', 'sentence', 'worker_0', 'worker_1', 'worker_2', 'worker_3', 'worker_4', 'worker_5', 'worker_6', 'worker_7', 'disagree', 'agree', 'neutral', 'guid', 'in_held_out_test', 'sentence_hi', 'sentence_bn']


,Unnamed: 0,round,batch,sent_id,sentence,worker_0,worker_1,worker_2,worker_3,worker_4,worker_5,worker_6,worker_7,disagree,agree,neutral,guid,in_held_out_test,sentence_hi,sentence_bn
0,0,1,0,s0,Global warming is a hoax.,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,NaN,NaN,NaN,1_0_s0,False,वैश्विक तापमान एक hax है.,গ্লোবাল গরম একটি হোকস।
1,1,1,0,s1,Alarming levels of sea level rise are predicte...,neutral,neutral,neutral,agrees,agrees,neutral,neutral,agrees,NaN,NaN,NaN,1_0_s1,False,समुद्र स्तर के अलार्म के स्तर अगले दशकों के दौ...,পরবর্তী দশক ধরে ফ্রিলিডিকে হুমকি দেবে নৌবাহিনী...
2,2,1,0,s2,"Over the past several years, the United States...",neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,NaN,NaN,NaN,1_0_s2,False,"पिछले कई सालों से, अमरीका ने व्यापार वृद्धि मे...","গত কয়েক বছর ধরে, যুক্তরাষ্ট্র ব্যবসায়িক বৃদ্..."
3,3,1,0,s3,Global warming is happening and it will be dan...,agrees,agrees,agrees,agrees,agrees,agrees,agrees,agrees,NaN,NaN,NaN,1_0_s3,False,विश्‍वव्यापी तापमान हो रहा है और यह मानव स्वास...,গ্লোবাল গরমের ঘটনা ঘটেছে এবং এটি মানবাধিকার ও ...
4,4,1,0,s4,Some icebergs are cute.,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,NaN,NaN,NaN,1_0_s4,False,कुछ कीड़े काट रहे हैं.,কিছু আইসবেরি খুবই সুন্দর।


Prepare labels and split the dataset

In [15]:
score_columns = ["agree", "disagree", "neutral"]

labelled_df = multilingual_df.dropna(
    subset=score_columns
).copy()

labelled_df["stance"] = labelled_df[
    score_columns
].idxmax(axis=1)

labelled_df["stance"] = labelled_df["stance"].map({
    "agree": "Favour",
    "disagree": "Against",
    "neutral": "Neutral"
})

train_val_df = labelled_df[
    labelled_df["in_held_out_test"] == False
].copy()

test_df = labelled_df[
    labelled_df["in_held_out_test"] == True
].copy()

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.10,
    random_state=42,
    stratify=train_val_df["stance"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining label distribution:")
print(train_df["stance"].value_counts())

print("\nValidation label distribution:")
print(val_df["stance"].value_counts())

print("\nTesting label distribution:")
print(test_df["stance"].value_counts())

Training samples: 1665
Validation samples: 185
Testing samples: 200

Training label distribution:
stance
Neutral    709
Favour     631
Against    325
Name: count, dtype: int64

Validation label distribution:
stance
Neutral    79
Favour     70
Against    36
Name: count, dtype: int64

Testing label distribution:
stance
Neutral    85
Favour     76
Against    39
Name: count, dtype: int64


Define label mapping

In [16]:
label2id = {
    "Against": 0,
    "Favour": 1,
    "Neutral": 2
}

id2label = {
    0: "Against",
    1: "Favour",
    2: "Neutral"
}

train_labels = train_df["stance"].map(label2id).values
val_labels = val_df["stance"].map(label2id).values
test_labels = test_df["stance"].map(label2id).values

print("Label mapping:", label2id)
print("Training labels shape:", train_labels.shape)
print("Validation labels shape:", val_labels.shape)
print("Testing labels shape:", test_labels.shape)

Label mapping: {'Against': 0, 'Favour': 1, 'Neutral': 2}
Training labels shape: (1665,)
Validation labels shape: (185,)
Testing labels shape: (200,)


Tokenize English, Hindi, and Bengali

In [17]:
# Prepare text
train_texts = {
    "English": train_df["sentence"].astype(str).tolist(),
    "Hindi": train_df["sentence_hi"].astype(str).tolist(),
    "Bengali": train_df["sentence_bn"].astype(str).tolist()
}

val_texts = {
    "English": val_df["sentence"].astype(str).tolist(),
    "Hindi": val_df["sentence_hi"].astype(str).tolist(),
    "Bengali": val_df["sentence_bn"].astype(str).tolist()
}

test_texts = {
    "English": test_df["sentence"].astype(str).tolist(),
    "Hindi": test_df["sentence_hi"].astype(str).tolist(),
    "Bengali": test_df["sentence_bn"].astype(str).tolist()
}

# Tokenize each language
train_encodings = {}
val_encodings = {}
test_encodings = {}

for language in ["English", "Hindi", "Bengali"]:

    train_encodings[language] = indicbert_tokenizer(
        train_texts[language],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    val_encodings[language] = indicbert_tokenizer(
        val_texts[language],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    test_encodings[language] = indicbert_tokenizer(
        test_texts[language],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    print(f"{language} tokenization completed!")

English tokenization completed!
Hindi tokenization completed!
Bengali tokenization completed!


Create PyTorch datasets

In [18]:
class StanceDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

    def __len__(self):
        return len(self.labels)


train_datasets = {}
val_datasets = {}
test_datasets = {}

for language in ["English", "Hindi", "Bengali"]:

    train_datasets[language] = StanceDataset(
        train_encodings[language], train_labels
    )

    val_datasets[language] = StanceDataset(
        val_encodings[language], val_labels
    )

    test_datasets[language] = StanceDataset(
        test_encodings[language], test_labels
    )

    print(f"{language}:")
    print("Training:", len(train_datasets[language]))
    print("Validation:", len(val_datasets[language]))
    print("Testing:", len(test_datasets[language]))

English:
Training: 1665
Validation: 185
Testing: 200
Hindi:
Training: 1665
Validation: 185
Testing: 200
Bengali:
Training: 1665
Validation: 185
Testing: 200


Define evaluation metrics

In [19]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),

        "precision": precision_score(
            labels, predictions,
            average="macro",
            zero_division=0
        ),

        "recall": recall_score(
            labels, predictions,
            average="macro",
            zero_division=0
        ),

        "f1": f1_score(
            labels, predictions,
            average="macro",
            zero_division=0
        )
    }

print("Evaluation metrics function defined successfully!")

Evaluation metrics function defined successfully!


# Load IndicBERT for **English**

In [20]:
indicbert_model = AutoModelForSequenceClassification.from_pretrained(
    indicbert_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("IndicBERT English classification model loaded!")
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

IndicBERT English classification model loaded!
GPU available: True


# Configure English training

In [21]:
training_args = TrainingArguments(
    output_dir="./indicbert_english_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,

    logging_steps=50,
    save_total_limit=1,

    report_to="none"
)

print("English training configuration ready!")

English training configuration ready!


Create IndicBERT Trainer

In [22]:
indicbert_trainer = Trainer(
    model=indicbert_model,

    args=training_args,

    train_dataset=train_datasets["English"],

    eval_dataset=val_datasets["English"],

    compute_metrics=compute_metrics,

    processing_class=indicbert_tokenizer
)

print("IndicBERT English Trainer created successfully!")

IndicBERT English Trainer created successfully!


# Train IndicBERT on English

In [23]:
indicbert_train_result = indicbert_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.874100,0.806263,0.681081,0.733913,0.615679,0.619199
2,0.705733,0.736874,0.729730,0.735057,0.714674,0.713330
3,0.437139,0.707267,0.718919,0.720958,0.702980,0.707310


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Evaluate on the official English test set

In [24]:
english_results = indicbert_trainer.predict(
    test_datasets["English"]
)

english_predictions = np.argmax(
    english_results.predictions,
    axis=1
)

print("English IndicBERT Test Results")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(test_labels, english_predictions))

print("Macro F1:",
      f1_score(test_labels, english_predictions, average="macro"))

print("\nClassification Report:\n")

print(classification_report(
    test_labels,
    english_predictions,
    target_names=["Against", "Favour", "Neutral"],
    zero_division=0
))

English IndicBERT Test Results
--------------------------------
Accuracy: 0.735
Macro F1: 0.725

Classification Report:

              precision    recall  f1-score   support

     Against       0.66      0.69      0.68        39
      Favour       0.73      0.80      0.76        76
     Neutral       0.79      0.69      0.74        85

    accuracy                           0.73       200
   macro avg       0.72      0.73      0.72       200
weighted avg       0.74      0.73      0.73       200



In [25]:
indicbert_results = []

indicbert_results.append({
    "Language": "English",
    "Model": "IndicBERT v2",
    "Accuracy": accuracy_score(
        test_labels, english_predictions
    ),
    "Macro F1": f1_score(
        test_labels,
        english_predictions,
        average="macro"
    )
})

print(pd.DataFrame(indicbert_results))

  Language         Model  Accuracy  Macro F1
0  English  IndicBERT v2     0.735     0.725


# Load IndicBERT for **Hindi**

In [26]:
import gc

# Release English model and trainer
del indicbert_trainer
del indicbert_model

gc.collect()
torch.cuda.empty_cache()

# Load fresh pretrained IndicBERT
indicbert_model = AutoModelForSequenceClassification.from_pretrained(
    indicbert_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Fresh IndicBERT Hindi model loaded!")
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

Fresh IndicBERT Hindi model loaded!
GPU available: True


Configure Hindi training

In [27]:
hindi_training_args = TrainingArguments(
    output_dir="./indicbert_hindi_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,

    logging_steps=50,
    save_total_limit=1,

    report_to="none"
)

print("Hindi training configuration ready!")

Hindi training configuration ready!


Create Hindi Trainer

In [28]:
hindi_trainer = Trainer(
    model=indicbert_model,
    args=hindi_training_args,
    train_dataset=train_datasets["Hindi"],
    eval_dataset=val_datasets["Hindi"],
    compute_metrics=compute_metrics,
    processing_class=indicbert_tokenizer
)

print("IndicBERT Hindi Trainer created successfully!")

IndicBERT Hindi Trainer created successfully!


# Train IndicBERT on Hindi

In [29]:
hindi_train_result = hindi_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.983063,0.909374,0.594595,0.426086,0.497770,0.441113
2,0.889205,0.873539,0.610811,0.602652,0.550747,0.540455
3,0.691552,0.838221,0.627027,0.610603,0.584264,0.587222


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Evaluate Hindi on the official test set

In [30]:
hindi_results = hindi_trainer.predict(
    test_datasets["Hindi"]
)

hindi_predictions = np.argmax(
    hindi_results.predictions,
    axis=1
)

print("Hindi IndicBERT Test Results")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(test_labels, hindi_predictions))

print("Macro F1:",
      f1_score(test_labels, hindi_predictions, average="macro"))

print("\nClassification Report:\n")

print(classification_report(
    test_labels,
    hindi_predictions,
    target_names=["Against", "Favour", "Neutral"],
    zero_division=0
))

Hindi IndicBERT Test Results
--------------------------------
Accuracy: 0.64
Macro F1: 0.5822539802835369

Classification Report:

              precision    recall  f1-score   support

     Against       0.58      0.28      0.38        39
      Favour       0.70      0.66      0.68        76
     Neutral       0.61      0.79      0.69        85

    accuracy                           0.64       200
   macro avg       0.63      0.58      0.58       200
weighted avg       0.64      0.64      0.62       200



# Save Hindi results

In [31]:
indicbert_results.append({
    "Language": "Hindi",
    "Model": "IndicBERT v2",
    "Accuracy": accuracy_score(
        test_labels, hindi_predictions
    ),
    "Macro F1": f1_score(
        test_labels,
        hindi_predictions,
        average="macro"
    )
})

print(pd.DataFrame(indicbert_results))

  Language         Model  Accuracy  Macro F1
0  English  IndicBERT v2     0.735  0.725000
1    Hindi  IndicBERT v2     0.640  0.582254


# Clear Hindi model and load a fresh Bengali model

In [32]:
import gc

# Release Hindi model and trainer
del hindi_trainer
del indicbert_model

gc.collect()
torch.cuda.empty_cache()

# Load fresh pretrained IndicBERT
indicbert_model = AutoModelForSequenceClassification.from_pretrained(
    indicbert_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Fresh IndicBERT Bengali model loaded!")
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

Fresh IndicBERT Bengali model loaded!
GPU available: True


Configure Bengali training

In [33]:
bengali_training_args = TrainingArguments(
    output_dir="./indicbert_bengali_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,

    logging_steps=50,
    save_total_limit=1,

    report_to="none"
)

print("Bengali training configuration ready!")

Bengali training configuration ready!


Create Bengali Trainer

In [34]:
bengali_trainer = Trainer(
    model=indicbert_model,
    args=bengali_training_args,
    train_dataset=train_datasets["Bengali"],
    eval_dataset=val_datasets["Bengali"],
    compute_metrics=compute_metrics,
    processing_class=indicbert_tokenizer
)

print("IndicBERT Bengali Trainer created successfully!")

IndicBERT Bengali Trainer created successfully!


# Train IndicBERT on Bengali

In [35]:
bengali_train_result = bengali_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.968504,0.898149,0.583784,0.399020,0.488788,0.429903
2,0.871739,0.803210,0.594595,0.561427,0.527466,0.510361
3,0.674525,0.782021,0.616216,0.563728,0.543259,0.529377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Evaluate Bengali on the official test set

In [36]:
bengali_results = bengali_trainer.predict(
    test_datasets["Bengali"]
)

bengali_predictions = np.argmax(
    bengali_results.predictions,
    axis=1
)

print("Bengali IndicBERT Test Results")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(test_labels, bengali_predictions))

print("Macro F1:",
      f1_score(test_labels, bengali_predictions, average="macro"))

print("\nClassification Report:\n")

print(classification_report(
    test_labels,
    bengali_predictions,
    target_names=["Against", "Favour", "Neutral"],
    zero_division=0
))

Bengali IndicBERT Test Results
--------------------------------
Accuracy: 0.62
Macro F1: 0.5882651319319124

Classification Report:

              precision    recall  f1-score   support

     Against       0.74      0.36      0.48        39
      Favour       0.69      0.58      0.63        76
     Neutral       0.56      0.78      0.65        85

    accuracy                           0.62       200
   macro avg       0.66      0.57      0.59       200
weighted avg       0.64      0.62      0.61       200



# Save Bengali results

In [37]:
indicbert_results.append({
    "Language": "Bengali",
    "Model": "IndicBERT v2",
    "Accuracy": accuracy_score(
        test_labels, bengali_predictions
    ),
    "Macro F1": f1_score(
        test_labels,
        bengali_predictions,
        average="macro"
    )
})

indicbert_comparison = pd.DataFrame(indicbert_results)

print("IndicBERT v2 - Final Results")
print("--------------------------------")
print(indicbert_comparison)

IndicBERT v2 - Final Results
--------------------------------
  Language         Model  Accuracy  Macro F1
0  English  IndicBERT v2     0.735  0.725000
1    Hindi  IndicBERT v2     0.640  0.582254
2  Bengali  IndicBERT v2     0.620  0.588265
